## Software Design Patterns in Python

### Day 2: Creational Patterns, Accessors & AOP
#### Module 2: The Python Data Model & Protocols
 - **Descriptor Protocol:** Implementing `__get__`, `__set__`, and `__getattr__` / `__setattr__` / `__delattr__` (The foundation of Python properties).
 - **Property accessors:** Using the builtin `@property` accessors.
 - **Context Management:** Creating custom context managers with `__enter__` and `__exit__`.
 - **Callable Objects:** Using `__call__` to make instances behave like functions.
 - **Review:** How these hooks replace "boilerplate" code found in Java/C++ patterns.

#### Module 3: Decorators & AOP (Aspect Oriented Programming)
 - **Decorator Deep Dive:** Function vs. Class decorators.
 - **Metadata Preservation:** Using `functools.wraps` to preserve introspection.
 - **Pattern Application:** Logging, Caching (Memoization), and Access Control (Proxying).
 - **Parameterized Decorators:** Building decorators that accept configuration arguments.

#### Module 4: Pythonic Creational Patterns
 - **Singleton:** Four implementation styles (Module-level, Decorator, Metaclass, Monostate).
 - **Factory Method & Abstract Factory:** Leveraging dynamic typing and classes-as-objects.
 - **Builder:** Implementing fluent interfaces and method chaining.
 - **Prototype:** Effective use of `copy.deepcopy` vs custom cloning methods.



---

#### Exercise for today (Day 1):
 Create a command execution pipeline.
```
 import sys
 Run("ls") | Run("wc -l") > sys.stdout 
```

The above code should mimic the unix pipeline: ls | wc -l
For running commands and capturing output: use `subprocess.Popen`


Solution to the Run() process exercise

In [9]:
import subprocess
import sys

class Run:
    def __init__(self, cmd):
        self.cmd = cmd
    
    def __or__(self, other):
        return Run(f"{self.cmd} | {other.cmd}")
    
    def __gt__(self, out):
        subprocess.run(self.cmd, shell=True, stdout=out, stderr=out)

    def run(self):
        self > sys.stdout

if __name__ == "__main__":
    Run("ls") | Run("wc -l") > sys.stdout
    Run("ls -lrt") > sys.stdout
    Run("ls -lrt").run()

#### Another exercise: Implement a Queue with intuitive operator support to add / remove elements.

In [ ]:
from collections import deque

class Queue:
    def __init__(self, *args, **kwargs):
        self.queue = deque(*args, **kwargs)

    # TODO: Implement the '>>' operator to enqueue an element to the queue
    
    # TODO: Implement the '<<' operator to dequeue an element from the queue

    def __str__(self):
        return f"Queue({list(self.queue)})"

if __name__ == '__main__':
    q = Queue()
    2 >> q  # Enqueue element [2]
    3 >> q  # [3, 2]
    4 >> q  # [4, 3, 2]

    q << 10 # Enqueue element 10 [4, 3, 2, 10]
    q << 11 # [4, 3, 2, 10, 11]
    q << 12 # [4, 3, 2, 10, 11, 12]

    r = Queue()
    q >> r # Dequeue element from q and enqueue to r [12], q=[4, 3, 2, 10, 11]
    q >> r # [11, 12], q=[4, 3, 2, 10]
    q >> r # [10, 11, 12], q=[4, 3, 2]

    print(q)
    print(r)


In [14]:
from collections import deque

class Queue:
    def __init__(self, *args, **kwargs):
        self.queue = deque(*args, **kwargs)

    def __rrshift__(self, other):
        self.queue.appendleft(other)
        return self
    
    def __lshift__(self, other):
        self.queue.append(other)
        return self
 
    def __rshift__(self, other):
        if type(other) is type(self):
            other.queue.appendleft(self.queue.pop())
      
    def __repr__(self):
        return f"Queue({list(self.queue)})"

q = Queue()
2 >> q  # Enqueue element [2]
print(q)
3 >> q  # [3, 2]
print(q)
4 >> q  # [4, 3, 2]
print(q)

q << 10 # Enqueue element 10 [4, 3, 2, 10]
print(q)
q << 11 # [4, 3, 2, 10, 11]

q << 12 # [4, 3, 2, 10, 11, 12]
print(q)

r = Queue()
print(f"{q=}, {r=}")
q >> r # Dequeue element from q and enqueue to r [12], q=[4, 3, 2, 10, 11]
print(f"{q=}, {r=}")
q >> r # [11, 12], q=[4, 3, 2, 10]
print(f"{q=}, {r=}")
q >> r # [10, 11, 12], q=[4, 3, 2]
print(f"{q=}, {r=}")



Queue([2])
Queue([3, 2])
Queue([4, 3, 2])
Queue([4, 3, 2, 10])
Queue([4, 3, 2, 10, 11, 12])
q=Queue([4, 3, 2, 10, 11, 12]), r=Queue([])
q=Queue([4, 3, 2, 10, 11]), r=Queue([12])
q=Queue([4, 3, 2, 10]), r=Queue([11, 12])
q=Queue([4, 3, 2]), r=Queue([10, 11, 12])


---

#### Type conversion / coercion of Python objects

In [ ]:
class User:
    def __init__(self, name):
        self.name = name

    def __str__(self):
        return f"<User: name='{self.name}'>"

    def __repr__(self):
        return f"User(name='{self.name}')"  # Official representation (should mimic python expression)

u1 = User("Alice")
u2 = User("Bob")
print(u1) # str(u1)
print(u2)
u1  # repr(u1)

<User: name='Alice'>
<User: name='Bob'>


User(name='Alice')

In [ ]:
# Real-world example of __str__ and __repr__:
import requests

res = requests.get("https://python.org/")
res

<Response [200]>

In [ ]:
def foo(): pass

foo

<function __main__.foo()>

#### Implementing boolean context of a custom python object

In [21]:
class Connection:
    def __init__(self):
        self.status = "Disconnected"

    def connect(self):
        self.status = "Connected"

    def __str__(self):
        return f"Connection(status='{self.status}')"
    
    def __bool__(self):
        return self.status == "Connected"

c = Connection()
print(c)
print(bool(c))
c.connect()
print(c)
print(bool(c))

Connection(status='Disconnected')
False
Connection(status='Connected')
True


In [23]:
class Stats:
    def __int__(self):
        return 42

s = Stats()
print(str(s), bool(s))
print(int(s))

<__main__.Stats object at 0x10a231040> True
42


#### Implementing support for int() and float() conversions

In [31]:
class Stats:
    def __init__(self, temperature, humidity):
        self.temperature = temperature
        self.humidity = humidity

    def __int__(self):
        return int(self.humidity)
    
    def __float__(self):
        return float(self.temperature)
    
    def __add__(self, other):
        if isinstance(other, Stats):
            return Stats(self.temperature + other.temperature, self.humidity + other.humidity)
        elif isinstance(other, int) :
            return Stats(self.temperature, self.humidity + other)
        elif isinstance(other, float):
            return Stats(self.temperature + other, self.humidity)
        else:
            return NotImplemented

    __radd__ = __add__  # For commutative addition    

    def __str__(self):
        return f"Stats<temperature: {self.temperature}, humidity: {self.humidity}>"
    
    def __repr__(self):
        return f"Stats(temperature={self.temperature}, humidity={self.humidity})"

s = Stats(22.5, 60)
print(s)
print(int(s))
print(float(s))
# NOTE: int() and float() output can seem ambiguous, use with caution
print(s + 10)
print(10 + s)

print(s + 5.5)

Stats<temperature: 22.5, humidity: 60>
60
22.5
Stats<temperature: 22.5, humidity: 70>
Stats<temperature: 22.5, humidity: 70>
Stats<temperature: 28.0, humidity: 60>


In [32]:
from collections import deque

class Queue:
    def __init__(self, *args, **kwargs):
        self.queue = deque(*args, **kwargs)

    def __rrshift__(self, v): # 2 >> q
        self.queue.appendleft(v)

    def __lshift__(self, v): # q << 10
        self.queue.append(v)

    def __rshift__(self, dest): # q >> r
        if type(dest) is type(self):
            dest.queue.appendleft(self.queue.pop())
        elif hasattr(dest, 'appendleft'):
            dest.appendleft(self.queue.pop())
        elif hasattr(dest, 'insert'):
            dest.insert(0, self.queue.pop())
        elif dest is None:
            self.queue.pop()
        else:
            raise TypeError("Destination must be a Queue, deque, list, or None")
        return dest

    def __repr__(self):
        return f"Queue({list(self.queue)})"

In [35]:
q1 = Queue([10, 20, 30, 40])
print(q1)
q1 << 60
print(q1)
70 >> q1
print(q1)

Queue([10, 20, 30, 40])
Queue([10, 20, 30, 40, 60])
Queue([70, 10, 20, 30, 40, 60])


In [36]:
q1[0]

TypeError: 'Queue' object is not subscriptable

In [37]:
for item in q1:
    print(item)

TypeError: 'Queue' object is not iterable

#### Python uses "Ducktyping" to infer the object's features and capabilities

#### Three traits of collections in Python

1. Length: ```len(obj)``` should return a positive integer, not raise an exception.
2. Searchability: ```item in obj``` should return True / False, not raise an exception.
3. Iterability: ```iter(obj)``` should return an iterator, not raise an exception.


In [41]:
a = [10, 20, 30, 40]
print(len(a), 30 in a, 50 in a, iter(a))
for v in a:
    print(v)

4 True False <list_iterator object at 0x10a21fdf0>
10
20
30
40


In [48]:
ins = open("sample.txt")
ins

<_io.TextIOWrapper name='sample.txt' mode='r' encoding='UTF-8'>

In [45]:
print("test" in ins)
print(iter(ins))

False
<_io.TextIOWrapper name='sample.txt' mode='r' encoding='UTF-8'>


In [46]:
len(ins)

TypeError: object of type '_io.TextIOWrapper' has no len()

In [49]:
for line in ins:
    print(line)

this is line 1

this is line 2

this is another line

this is another boring line


In [47]:
ins.close()

In [51]:
r = range(10)
print(r, type(r))

range(0, 10) <class 'range'>


In [54]:
len(r), 5 in r, iter(r)

(10, True, <range_iterator at 0x10a2339c0>)

In [68]:
from collections import deque

class Queue:
    def __init__(self, *args, **kwargs):
        self.queue = deque(*args, **kwargs)

    def __rrshift__(self, v): # 2 >> q
        self.queue.appendleft(v)

    def __lshift__(self, v): # q << 10
        self.queue.append(v)

    def __rshift__(self, dest): # q >> r
        if type(dest) is type(self):
            dest.queue.appendleft(self.queue.pop())
        else:
            raise TypeError("Destination must be a Queue")
        return dest

    def __repr__(self):
        return f"Queue({list(self.queue)})"
    
    def __getitem__(self, index):
        try:
            return self.queue[index]
        except IndexError:
            raise IndexError("Index out of range")
        
    def __len__(self):
        return len(self.queue)

q = Queue([10, 20, 30, 40, 50, 60, 70, 80])
q[2] # Queue.__getitem__(q, 2)
for item in q:
    print(item)

30 in q
len(q)

10
20
30
40
50
60
70
80


8

In [80]:
# A very simple example of a "generator pattern"
class Square:
    def __init__(self, limit):
        self.limit = limit

    def __getitem__(self, index):
        if index < self.limit:
            return index * index
        else:
            raise IndexError("Index out of range")
    
    def resize(self, new_limit):
        self.limit = new_limit

    def __len__(self):
        return self.limit
        
s = Square(10)

#for i in s:
#    print(i)

#print(5 in s)
print(len(s))
s.resize(15)
print(len(s))

10
15


In [98]:
class Users:
    def __init__(self):
        self.users = {}

    def __setitem__(self, name, role):
        self.users[name.lower()] = role

    def __getitem__(self, name):
        return self.users[name.lower()].title()
    
users = Users()
users["Alice"] = "admin"  # users.__setitem__("Alice", "admin") -> Users.__setitem__(users, "Alice", "admin")
users["Bob"] = "staff"
users["Charlie"] = "guest"

print(users["alice"], users["BOB"], users["ALiCE"])
iter(users)

Admin Staff Admin


In [117]:
class UserIterator:
    def __init__(self, users):
        self.users = iter(users.users.items())
       
    def __next__(self):
        return next(self.users)
    
class Users:
    def __init__(self):
        self.users = {}

    def __setitem__(self, name, role):
        self.users[name.lower()] = role

    def __getitem__(self, name):
        return self.users[name.lower()].title()
    
    def __iter__(self):
        return UserIterator(self)
    

users = Users()
users["Alice"] = "admin"  # users.__setitem__("Alice", "admin") -> Users.__setitem__(users, "Alice", "admin")
users["Bob"] = "staff"
users["Charlie"] = "guest"

print(users["alice"], users["BOB"], users["ALiCE"])
iter(users)

for user, role in users:
    print(user, role)

Admin Staff Admin
alice admin
bob staff
charlie guest


In [116]:
### The Iterator Protocol in Python

a = 1000

for item in a:
    print(item)

iterator = iter(a)
print(iterator)
try:
    while True:
        item = next(iterator)
        print(item) # For loop body
except StopIteration:
    pass

TypeError: 'int' object is not iterable

In [108]:
next(iterator)

StopIteration: 

In [83]:
from queue import Queue
q = Queue(maxsize=20)

q.maxsize

20

In [95]:
q.put(10)
q.qsize()

7

----

### Accessor pattern

In [125]:
class User:
    def __init__(self, name):
        self.name = name

u = User("Alice")
print(u.name)    

u.name = "Bob"
print(u.name)
u.name = 456
print(u.name)

#print(u.place)
u.place = "New York"  # Monkey patching: Adding an attribute to an instance at runtime
print(u.place)

del u.name
print(u.name)

Alice
Bob
456
New York


AttributeError: 'User' object has no attribute 'name'

In [129]:
class User: pass

u = User()

#u.name = "Alice" # -> setattr(u, "name", "Alice")

setattr(u, "name", "Alice") # Dynamic atrribute setter
print(u.name)
getattr(u, "name") # Dynamic attribute getter

# del u.name -> delattr(u, "name")
delattr(u, "name")

hasattr(u, "name")

Alice


False

In [ ]:
hasattr(u, "__iter__")

False

In [137]:
class User:
    def __getattr__(self, name):
        return f"Attribute '{name}' accessed"

u = User()
#u.name
getattr(u, "name") # u.__getattr__("name")
print(u.name)
print(u.role)
print(u.place)

Attribute 'name' accessed
Attribute 'role' accessed
Attribute 'place' accessed


In [141]:
class User:
    def __getattr__(self, name):
        return f"Attribute '{name}' accessed"

u = User()
print(u.name)
u.name = "John"
print(u.name)
print(u.role)
u.role = "Admin"
print(u.role)

Attribute 'name' accessed
John
Attribute 'role' accessed
Admin


In [142]:
class User:
    def __getattribute__(self, name):
        return f"Attribute '{name}' accessed"

u = User()
print(u.name)
u.name = "John"
print(u.name)
print(u.role)
u.role = "Admin"
print(u.role)

Attribute 'name' accessed
Attribute 'name' accessed
Attribute 'role' accessed
Attribute 'role' accessed


In [ ]:
class User:
    def __setattr__(self, name, value):
        print(f"Setting attribute '{name}' to '{value}'")
        super().__setattr__(name, value)  # To actually set the attribute
        #self.__dict__[name] = value  # Bypass __setattr__ to avoid infinite recursion

u = User()
u.name = "John"
print(u.name)

u.name = "Alice"

Setting attribute 'name' to 'John'
John
Setting attribute 'name' to 'Alice'


In [165]:
class User:
    def __init__(self, name):
        self.name = name

    def __setattr__(self, name, value):
        if type(value) is not str or not value.strip().isalpha():
            raise ValueError("Name must be a non-empty string")
        super().__setattr__(name, value)

u = User("John")
print(u.name)

u.name = "JohnDoe"
print(u.name)

John
JohnDoe


In [177]:
class User:
    def __init__(self, name):
        self.__name = name

    def __str__(self):
        return f"<User: name='{self.__name}'>"

    def __repr__(self):
        return f"User(name='{self.name}')"  # Official representation (should mimic python expression)
    
    @property
    def name(self):
        print("Accessing name property")
        return self.__name
    
    @name.setter
    def name(self, value):
        if not isinstance(value, str) or not value.strip().isalpha():
            raise ValueError("Name must be a non-empty string")
        print("Setting name property")
        self.__name = value


    @name.deleter
    def name(self):
        print("Deleting name property")
    

u = User("Alice")
print(u.name)

u.name = "John"
del u.name

Accessing name property
Alice
Setting name property
Deleting name property


In [175]:
from threading import Thread

t = Thread(target=lambda: print("Hello from thread!"))
t.start()
print(t.daemon)
t.daemon = True
print(t.daemon)

Hello from thread!
False


RuntimeError: cannot set daemon status of active thread

---

### The State Pattern

In [182]:
class Accumulator:
    def __init__(self):
        self.value = 0

    def __call__(self, x=0):
        self.value += x
        print(f"Current value: {self.value}")
        return self.value
    

a = Accumulator()
a(10)
a(20)
a(30)

Current value: 10
Current value: 30
Current value: 60


60

In [184]:
class LineHistory:
    def __init__(self):
        self.lines = []

    def __call__(self, line=None):
        if line is not None:
            self.lines.append(line)
            print(f"Line added: {line}")
            return line
        else:
            print("Line history:")
            for idx, l in enumerate(self.lines, 1):
                print(f"{idx}: {l}")
            return self.lines
        
history = LineHistory()
history("First line")
history("Second line")
history("Third line")
print("---")
history()
    


Line added: First line
Line added: Second line
Line added: Third line
---
Line history:
1: First line
2: Second line
3: Third line


['First line', 'Second line', 'Third line']

----

### Creational Patterns

1. Singleton Pattern
2. Factory Method
3. Abstract Factory Pattern
4. Prototype Pattern
5. Builder Pattern


---

### The Singleton Pattern



In [186]:
class World:
    def get_population(self):
        return 8_300_000_000
    
w1 = World()
print(w1.get_population())
w2 = World()
print(w2.get_population())
print(id(w1), id(w2))

8300000000
8300000000
4465003600 4465012240


In [188]:
class World:
    def __new__(cls):
        if not hasattr(cls, "_instance"):
            cls._instance = super().__new__(cls)
        return cls._instance
    
    def get_population(self):
        return 8_300_000_000
    
w1 = World()
print(w1.get_population())
w2 = World()
print(w2.get_population())
print(id(w1), id(w2))

w3 = World()
print(id(w1), id(w2), id(w3))

8300000000
8300000000
4465044240 4465044240
4465044240 4465044240 4465044240


In [189]:
class World:
    def __new__(cls):
        if not hasattr(cls, "_instance"):
            cls._instance = super().__new__(cls)
        return cls._instance
    
    def __init__(self):
        print("Initializing World instance")

    def get_population(self):
        return 8_300_000_000
    
w1 = World()
print(w1.get_population())
w2 = World()
print(w2.get_population())
print(id(w1), id(w2))

w3 = World()
print(id(w1), id(w2), id(w3))

Initializing World instance
8300000000
Initializing World instance
8300000000
4468584528 4468584528
Initializing World instance
4468584528 4468584528 4468584528


In [195]:
def singleton(cls):
    instances = {}
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    get_instance.__qualname__ = f"<@singleton: {cls}>"
    return get_instance

@singleton
class World:
    def __init__(self):
        print("Initializing World instance")

    def get_population(self):
        return 8_300_000_000

print(World)

w1 = World()
w2 = World()
w3 = World()
print(id(w1), id(w2), id(w3))

<function <@singleton: <class '__main__.World'>> at 0x10ae7e7a0>
Initializing World instance
4468017392 4468017392 4468017392


#### NOTE:

Though Singleton Pattern using classes  are common in other programming languages (C++/Java/C#), it is considered
un-pythonic!

The pythonic approach to implement singleton pattern is simply using a module!


For implementing singleton pattern, the clean pythonic approach should be to create a module.

---

### Factory Method

In [ ]:
path1 = "./sample.txt"
path2 = "https://python.org/"

def url_reader(url):
    import requests
    response = requests.get(url)
    return response.text

def file_reader(path):
    with open(path, 'r') as f:
        return f.read()

def reader(path):
    if path.startswith("http://") or path.startswith("https://"):
        return url_reader(path)
    else:
        return file_reader(path)

contents = reader(path1)
print(contents)
contents = reader(path2)
print(contents)


this is line 1
this is line 2
this is another line
this is another boring line
<!doctype html>
<html class="no-js" lang="en" dir="ltr">

<head>
    <script defer
            file-types="bz2,chm,dmg,exe,gz,json,msi,msix,pdf,pkg,tgz,xz,zip"
            data-domain="python.org"
            src="https://analytics.python.org/js/script.file-downloads.outbound-links.js"></script>

    <meta charset="utf-8">

    <link rel="prefetch" href="//ajax.googleapis.com/ajax/libs/jquery/1.8.2/jquery.min.js">
    <link rel="prefetch" href="//ajax.googleapis.com/ajax/libs/jqueryui/1.12.1/jquery-ui.min.js">

    <meta name="application-name" content="Python.org">
    <meta name="apple-mobile-web-app-title" content="Python.org">
    <meta name="apple-mobile-web-app-capable" content="yes">
    <meta name="apple-mobile-web-app-status-bar-style" content="black">

    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <meta name="HandheldFriendly" content="True">
    <meta name="format-

In [ ]:
# A simple factory method implementation using functions.

path1 = "./sample.txt"
path2 = "https://python.org/"

def url_reader(url):
    import requests
    response = requests.get(url)
    return response.text

def file_reader(path):
    with open(path, 'r') as f:
        return f.read()

def reader(path):
    if path.startswith("http://") or path.startswith("https://"):
        return lambda: url_reader(path)
    else:
        return lambda: file_reader(path)

r1 = reader(path1)
r2 = reader(path2)

r2()


'<!doctype html>\n<html class="no-js" lang="en" dir="ltr">\n\n<head>\n    <script defer\n            file-types="bz2,chm,dmg,exe,gz,json,msi,msix,pdf,pkg,tgz,xz,zip"\n            data-domain="python.org"\n            src="https://analytics.python.org/js/script.file-downloads.outbound-links.js"></script>\n\n    <meta charset="utf-8">\n\n    <link rel="prefetch" href="//ajax.googleapis.com/ajax/libs/jquery/1.8.2/jquery.min.js">\n    <link rel="prefetch" href="//ajax.googleapis.com/ajax/libs/jqueryui/1.12.1/jquery-ui.min.js">\n\n    <meta name="application-name" content="Python.org">\n    <meta name="apple-mobile-web-app-title" content="Python.org">\n    <meta name="apple-mobile-web-app-capable" content="yes">\n    <meta name="apple-mobile-web-app-status-bar-style" content="black">\n\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <meta name="HandheldFriendly" content="True">\n    <meta name="format-detection" content="telephone=no">\n\n    <script async\n